## Import libraries and define necessary things

In [1]:
import gdown

gdown.download(id="1i_64NJkjlxjl7NMQfSQmjsvkok-uA7aQ", output="storyteller.pth")

Downloading...
From (original): https://drive.google.com/uc?id=1i_64NJkjlxjl7NMQfSQmjsvkok-uA7aQ
From (redirected): https://drive.google.com/uc?id=1i_64NJkjlxjl7NMQfSQmjsvkok-uA7aQ&confirm=t&uuid=b92f8290-6e51-41bd-9047-5e0346d4780d
To: /kaggle/working/storyteller.pth
100%|██████████| 137M/137M [00:01<00:00, 76.1MB/s]


'storyteller.pth'

In [2]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from tqdm.auto import tqdm

In [3]:
SAVE_PATH = "storyteller.pth"
SEQ_SIZE = 150

device = "cuda" if torch.cuda.is_available() else "cpu"

## Get the data and prepare the data

In [4]:
from datasets import load_dataset

# Load a subset for faster training
ds = load_dataset("eminorhan/tinystories", "10M_1", split="validation")  # 10M stories subset

README.md: 0.00B [00:00, ?B/s]

10M_1/train-00000-of-00001.parquet:   0%|          | 0.00/26.0M [00:00<?, ?B/s]

10M_1/validation-00000-of-00001.parquet:   0%|          | 0.00/11.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62345 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/27635 [00:00<?, ? examples/s]

In [5]:
print(ds[0]["text"])

u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.


In [6]:
len(ds)

27635

In [7]:
all_text = " ".join(ds[:1000]["text"]) # use 1st 1000 stories
chars = sorted(set(all_text))

In [8]:
# build the vocabulary

stoi = {char: i for i, char in enumerate(chars, 1)}
stoi["*UNK*"] = 0
iots = {v: k for k, v in stoi.items()}

vocab_size = len(stoi)
print(f"{vocab_size = }")

encode = lambda x: [stoi.get(char, 0) for char in x]
decode = lambda x: "".join([iots.get(i, 0) for i in x])

vocab_size = 72


In [9]:
encode("Encoded successfully!")

[20,
 54,
 43,
 55,
 44,
 45,
 44,
 2,
 59,
 61,
 43,
 43,
 45,
 59,
 59,
 46,
 61,
 52,
 52,
 65,
 3]

In [10]:
encoded_token = torch.tensor(encode(all_text))

In [11]:
class Data(Dataset):
    def __init__(self, data, block_size=128):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

In [12]:
data = Data(encoded_token, block_size=SEQ_SIZE)

## Build **Transformer** Model

In [13]:
class TransformerBlock(nn.Module):
    """
    Desc: forward with MultiheadAttention and then a dense layer
    Args:
      emb_size -> embedding size
      ff_hidden_size -> number of hidden stage in the dense layer
      num_heads -> number of heads in multi attention
    """
    def __init__(self, emb_size: int, ff_hidden_size: int=4, num_heads: int=4):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            emb_size,
            num_heads,
            dropout=0.1,
            batch_first=True
        )
        self.ln1 = nn.LayerNorm(emb_size)

        self.ff = nn.Sequential(
            nn.Linear(emb_size, ff_hidden_size* emb_size),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(ff_hidden_size * emb_size, emb_size),
            nn.Dropout(0.1)
        )

        self.ln2 = nn.LayerNorm(emb_size)

    def forward(self, x):
        """
        Preserve the x as residual and then normalize x and apply attention.
        add residual with attention output.
        Pass it through the dense layer and do the same
        """
        T = x.shape[1]
        # Causal mask
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        # attention output
        residual = x
        x = self.ln1(x)
        attn_out, _ = self.attn(x, x, x, attn_mask=causal_mask)
        x = residual + attn_out

        # dense layer
        residual = x
        x = self.ln2(x)
        ff_out = self.ff(x)
        x = ff_out + residual

        return x

In [14]:
class GPTAI(nn.Module):
    def __init__(self,
        vocab_size: int,
        emb_size: int,
        num_heads: int,
        ff_hidden_size: int,
        max_seq_len: int=512,
        num_blocks: int=3
    ):
        """
        Desc: Stack transfomer blocks and combine token, positional embedding to predict the next token

        Args:
        vocab_size -> vocabulary size
        emb_size -> embedding size
        num_heads -> number of heads in multi head attention
        ff_hidden_size -> hidden stage in dense layer
        max_seq_len -> maximum sequence length for positional embedding
        num_blocks -> number of transformer stacks
        """
        super().__init__()

        # token embed
        self.token_emb = nn.Embedding(vocab_size, emb_size)
        self.pos_emb = nn.Embedding(max_seq_len, emb_size)

        # stack transfomer block for num_block times
        transformers = [TransformerBlock(
            emb_size=emb_size,
            ff_hidden_size=ff_hidden_size,
            num_heads=num_heads
        ) for _ in range(num_blocks)]

        self.transformers = nn.Sequential(*transformers)

        self.fc = nn.Linear(emb_size, vocab_size)

    def forward(self, x):
        """
        Do Positional and Token embedding then pass it to transformers and return output as the final layer
        """
        x = x.long()
        B, T = x.shape # batch, Sequence length
        # position
        pos = torch.arange(T, device=x.device) # 0 - (T-1) position

        pos_emb = self.pos_emb(pos)
        token_emb = self.token_emb(x)
        x = pos_emb + token_emb # add both token and positional embedding

        x = self.transformers(x)

        return self.fc(x)

In [15]:
model = GPTAI(
  vocab_size=vocab_size,
  emb_size=256,
  num_heads=2,
  ff_hidden_size=128,
  max_seq_len=SEQ_SIZE,
  num_blocks=2
).to(device)

summary(model, input_size=data[0][0].shape)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
         Embedding-1                  [-1, 256]          38,400
         Embedding-2             [-1, 150, 256]          18,432
         LayerNorm-3             [-1, 150, 256]             512
MultiheadAttention-4  [[-1, 150, 256], [-1, 150, 150]]               0
         LayerNorm-5             [-1, 150, 256]             512
            Linear-6           [-1, 150, 32768]       8,421,376
              GELU-7           [-1, 150, 32768]               0
           Dropout-8           [-1, 150, 32768]               0
            Linear-9             [-1, 150, 256]       8,388,864
          Dropout-10             [-1, 150, 256]               0
 TransformerBlock-11             [-1, 150, 256]               0
        LayerNorm-12             [-1, 150, 256]             512
MultiheadAttention-13  [[-1, 150, 256], [-1, 150, 150]]               0
        LayerNorm-14    

In [16]:
model.load_state_dict(torch.load(SAVE_PATH))

<All keys matched successfully>

## Train the model

In [17]:
def train_model(model: torch.nn.Module, train_data: Dataset, batch: int=32, epochs: int=10, lr: float=0.01, verbose: int=0, use_amp: bool=True):
  """ Train the model with the provided data of given batch, epochs, learning rate.
      Use AMP(Automatic mixed precision) if use_amp=True
  """

  if torch.cuda.device_count() > 1:
     model = nn.DataParallel(model)

  train_loader = DataLoader(train_data, batch_size=batch, pin_memory=True, shuffle=True)
  optimizer = torch.optim.Adam(model.parameters(), lr=lr)
  loss_fn = nn.CrossEntropyLoss()
  scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
  scheduler = torch.optim.lr_scheduler.OneCycleLR(
      optimizer,
      max_lr=lr,
      steps_per_epoch=len(train_loader),
      epochs=epochs
  )

  model.train()
  for epoch in range(epochs):
     total_loss = 0
     pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
     for i, (inp, tar) in enumerate(pbar):
        inp = inp.to(device, non_blocking=True)
        tar = tar.to(device, non_blocking=True)

        optimizer.zero_grad()

        # AMP: Autocast forward pass
        with torch.autocast("cuda", enabled=use_amp):
            out = model(inp)
            loss = loss_fn(out.view(-1, vocab_size), tar.view(-1))

        # AMP: Scaled backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        scaler.step(optimizer)
        scaler.update()

        scheduler.step()

        current_loss = loss.item()
        total_loss += current_loss
        pbar.set_postfix({"Loss": f"{current_loss:.4f}", "Avg Loss": f"{total_loss/(i+1):.4f}"})

     if verbose == 1:
        print(f"Epoch {epoch+1} completed - Average Loss: {total_loss/len(train_loader):.4f}")

In [18]:
train_model(
    model,
    data,
    batch=100,
    epochs=4,
    lr=5e-4,
    verbose=1
)

Epoch 1/4:   0%|          | 0/7936 [00:00<?, ?it/s]

Epoch 1 completed - Average Loss: 0.9583


Epoch 2/4:   0%|          | 0/7936 [00:00<?, ?it/s]

Epoch 2 completed - Average Loss: 0.3072


Epoch 3/4:   0%|          | 0/7936 [00:00<?, ?it/s]

Epoch 3 completed - Average Loss: 0.1539


Epoch 4/4:   0%|          | 0/7936 [00:00<?, ?it/s]

Epoch 4 completed - Average Loss: 0.1158


In [19]:
# save the model

torch.save(model.state_dict(), SAVE_PATH)

## Test: Generate text

In [20]:
@torch.no_grad()
def generate_text(model, start_str, length=500, temperature=1.0):
    model.eval()
    chars = encode(start_str)
    # Start with the initial sequence
    input_ids = torch.tensor(chars, device=device).unsqueeze(0)
    generated = start_str

    for _ in range(length):
        # We only pass the last 'seq_size' characters to the model
        cond_input = input_ids[:, -SEQ_SIZE:]

        logits = model(cond_input)
        # Focus on the last character's prediction
        logits = logits[:, -1, :] / temperature

        probs = torch.nn.functional.softmax(logits, dim=-1)
        next_char_idx = torch.multinomial(probs, num_samples=1)

        # Append the new character index to the running sequence
        input_ids = torch.cat((input_ids, next_char_idx), dim=1)

        generated += iots[next_char_idx.item()]

    return generated

In [21]:
prompt = "Once upon a time"
output = generate_text(model, prompt, length=500, temperature=0.8)
print(output)

Once upon a time, there was a sweet little girl named Mia. Mia loved to play with her toy animals. She had a toy cat and a toy dog. Mia liked to make them talk and play together. But one day, the cat and the dog had a quarrel.
Mia's mom made some dough to bake cookies for Mia. Mia was very excited to eat the sweet cookies. But Mia's mom told her she had to wait. Mia thought about the quarrel her toy cat and dog had. She wanted them to be friends again before eating the cookies.
Mia went to her room and made the


In [22]:
prompt = '"Good bye", he said.'
output = generate_text(model, prompt, length=300, temperature=1)
print(output)

"Good bye", he said. Soon, they were all back home safe and sound. One day, a boy named Tim found a bathrobe in his room. He put it on and felt very warm and cozy. Tim had a plan. He wanted to show his new bathrobe to his best friend, Sam.
Tim went to Sam's house and knocked on the door. Sam opened the door and saw Tom


In [23]:
prompt = "It was a night"
output = generate_text(model, prompt, length=300, temperature=0.75)
print(output)

It was a night, shiny ball. Tim and Mom went outside to see it. The rain had stopped. Tim touched the ball and it opened up. Inside was a friendly alien. The alien said, "Hello, I am Zog. I made the rain stop so we can play!" Tim, Mom, and Zog played outside all day, and they had lots of fun. One day, a little gi
